In [1]:

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

load_dotenv()

model1 = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

#🔹 What is a Runnable?

# Think of Runnable = anything that can run and give output

# In LangChain, these are all Runnables:

# PromptTemplate
# LLM (model)
# OutputParser

prompt1 = PromptTemplate(template='Generate short and simple notes from following text \n {text}',
                         input_variables=['text'])

prompt2 = PromptTemplate(template='Generate 5 short questions answer from following text \n text {text}' , 
                         input_variables=['text'])

prompt3 = PromptTemplate(template='Merge the provided notes and quiz into a single document \n notes {notes} and {quiz}' ,
                         input_variables=['notes' , 'quiz'])

parser = StrOutputParser()

#🔹 What is RunnableParallel?


# It runs multiple chains at the same time (parallel)

parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser ,
    'quiz': prompt2 | model1 | parser
})

merge_chain = prompt3 | model1 | parser
chain = parallel_chain | merge_chain

text = """
Machine Learning is a branch of Artificial Intelligence that allows systems to learn from data and improve over time without being explicitly programmed. 
There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. 
Supervised learning uses labeled data, unsupervised learning finds hidden patterns, and reinforcement learning learns through rewards and penalties.
"""

result = chain.invoke({'text':text})
print(result)

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here's the merged document combining your notes and quiz:

## Machine Learning Notes and Quiz

### Notes

*   **Machine Learning (ML):** Part of AI. Systems learn from data, improve without explicit programming.
*   **3 Main ML Types:**
    *   **Supervised:** Uses labeled data.
    *   **Unsupervised:** Finds hidden patterns.
    *   **Reinforcement:** Learns with rewards/penalties.

### Quiz

1.  **What is Machine Learning?**
    *   It's a branch of Artificial Intelligence that lets systems learn from data and improve without explicit programming.

2.  **How many main types of machine learning are there?**
    *   There are three main types.

3.  **What is the first type of machine learning mentioned?**
    *   Supervised learning.

4.  **What does unsupervised learning do?**
    *   It finds hidden patterns.

5.  **How does reinforcement learning learn?**
    *   It learns through rewards and penalties.


In [3]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

load_dotenv()

llm = ChatGroq(
        model_name="llama-3.1-8b-instant",
        temperature=0,
        max_tokens=1024,
        streaming=True        
        )

parser = StrOutputParser()
# StrOutputParser dont need partial_variable
prompt1 = PromptTemplate(
    template="""
    Generate one short joke about {topic}.

    Rules:
    - Must include the word "{topic}"
    - Max 2 lines
    - No explanations
    - No other topics

    Output only the joke.
    """,
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template="""
Explain the following joke in 2-3 short lines.

Rules:
- Keep it simple
- Keep it short
- Focus only on the joke meaning
- Do not add extra theory

Joke: {input}
""",
    input_variables=['input']
)

## RunnableSequence is same as '|' operator. | operator is just an another form of high level abstraction of RunnableSequence
chain = RunnableSequence(prompt1 , llm , prompt2 , llm , parser)
print(chain.invoke({'topic':'AI'}))

The AI went on a diet to lose some bytes, a play on words between digital data (bytes) and weight loss.


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough , RunnableSequence , RunnableParallel

load_dotenv()

llm = ChatGroq(
        model_name="llama-3.1-8b-instant",
        temperature=0,
        max_tokens=1024,
        streaming=True        
        )

parser = StrOutputParser()
# StrOutputParser dont need partial_variable

prompt1 = PromptTemplate(
    template="""
    Generate one short joke about {topic}.

    Rules:
    - Must include the word "{topic}"
    - Max 2 lines
    - No explanations
    - No other topics

    Output only the joke.
    """,
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template="""
Explain the following joke in 2-3 short lines.

Rules:
- Keep it simple
- Keep it short
- Focus only on the joke meaning
- Do not add extra theory

Joke: {input}
""",
    input_variables=['input']
)

# RunnablePassthrough Returns the input as-is, useful for preserving original output in parallel flows
joke_gen_chain = RunnableSequence(prompt1 , llm , parser)
parallel_chain = RunnableParallel({
    'joke':RunnablePassthrough(),
    'explanation':RunnableSequence(prompt2 , llm , parser)
})

final_chain = RunnableSequence(joke_gen_chain, parallel_chain)
print(final_chain.invoke({'topic':'cricket'}))

{'joke': 'Why did the cricket go to the doctor? \nBecause it had a ball ache.', 'explanation': 'The joke is about a cricket going to the doctor with a pain. \nThe phrase "ball ache" is a pun on the cricket ball and physical pain. \nIt\'s a play on words.'}


In [1]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough , RunnableSequence , RunnableParallel, RunnableLambda

load_dotenv()

llm = ChatGroq(
        model_name="llama-3.1-8b-instant",
        temperature=0,
        max_tokens=1024,
        streaming=True        
        )

parser = StrOutputParser()
# StrOutputParser dont need partial_variable

prompt = PromptTemplate(
    template="""
    Generate one short joke about {topic}.

    Rules:
    - Must include the word "{topic}"
    - Max 2 lines
    - No explanations
    - No other topics

    Output only the joke.
    """,
    input_variables=['topic']
)

def word_count(text):
    return len(text.split())

## RunnableLambda is a primitve that allows you to apply custom python functions within an AI pipeline.
first_chain = RunnableSequence(prompt, llm, parser)
second_chain = RunnableParallel({
    'joke':RunnablePassthrough(),
    'word_count':RunnableLambda(word_count)
})

merge_chain = RunnableSequence(first_chain , second_chain)
result = merge_chain.invoke({'topic':'School'})

final_result = """joke - {} \n word count - {}""".format(result['joke'],result['word_count'])
print(final_result)

joke - Why did the student bring a ladder to school? 
Because they wanted to reach their full potential. 
 word count - 17


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser , PydanticOutputParser
from langchain_core.runnables import RunnableParallel , RunnableBranch , RunnableLambda
from pydantic import BaseModel , Field
from typing import Literal

load_dotenv()

model1 = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

class Feedback(BaseModel):
    sentiment: Literal['positive' , 'negative'] = Field(description='Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(template='Classify the sentiment of the following feedback text into positive or negative \n {feedback} \n {format_instructions}',
                         input_variables=['feedback'] , partial_variables={'format_instructions': parser2.get_format_instructions()})

classifier_chain = prompt1 | model1 | parser2

prompt2 = PromptTemplate(template='Write an appropriate response to this positive feedback \n {feedback}' ,
                         input_variables=['feedback'])

prompt3 = PromptTemplate(template='Write an appropriate response to this negative feedback \n {feedback}' ,
                         input_variables=['feedback'])

#Input → sentiment classifier → structured output
    #   ↓
   #if positive → positive response
   #if negative → negative response
   #else → fallback

parser = StrOutputParser()

branch_chain = RunnableBranch((lambda x: x.sentiment == 'positive' , prompt2 | model1 | parser),
                              (lambda x: x.sentiment == 'negative' , prompt3 | model1 | parser),
                              RunnableLambda(lambda x: "Could not find sentiment"))

#🔹 7. RunnableLambda
# Used to run custom Python function inside chain

chain = classifier_chain | branch_chain
print(chain.invoke({'feedback':'This is a beautiful phone'}))

Here are several appropriate responses to positive feedback, categorized by tone and situation. Choose the one that best fits your personality and the context of the feedback:

**Short & Sweet (Good for quick acknowledgments):**

*   "Thank you so much!"
*   "That's wonderful to hear!"
*   "I really appreciate that!"
*   "Glad I could help!"
*   "Thanks for the kind words!"

**Slightly More Detailed & Enthusiastic:**

*   "Thank you! I'm so glad you're happy with [what you did/the result]."
*   "That's fantastic to hear! I'm really pleased that [specific aspect] worked out well."
*   "Thank you for the positive feedback! It means a lot."
*   "I'm so happy to hear that! Your feedback is greatly appreciated."
*   "That's wonderful! Thank you for taking the time to share your thoughts."

**Expressing Gratitude & Impact:**

*   "Thank you so much for your kind words! It's very motivating to hear that."
*   "I really appreciate you saying that. It makes my day!"
*   "Thank you! Knowing that